# SARWatch · Fase 4 · Fine-tuning de segmentación de agua con Sentinel-1 en Colab (GPU T4)

**Estado: plantilla no ejecutada.** Este cuaderno se ejecuta en Google Colab (Runtime → T4 GPU). Objetivo: entrenar un segmentador de agua abierta sobre Sen1Floods11 (banda C) con un backbone preentrenado (TerraMind o Clay vía TerraTorch), exportarlo a ONNX y evaluarlo luego en NISAR (banda L) con `scripts/evaluate_detection.py`.

Reglas: la métrica que cuenta es la de **bloques y fechas retenidos en NISAR**, no la de Sen1Floods11. El modelo entra al visor solo si supera a LightGBM (F1 0,83 / 0,80 en Pantanal). La brecha banda C → banda L debe medirse, no asumirse.

In [ ]:
!nvidia-smi -L
!pip -q install terratorch==1.1 rasterio onnx onnxruntime
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Datos: Sen1Floods11 (hand-labelled split)
Descarga desde el bucket público (`gs://sen1floods11`). Usamos VV/VH en dB y las etiquetas manuales; se excluyen píxeles -1 (sin etiqueta).

In [ ]:
!gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand /content/s1
!gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand /content/labels
!gsutil -m -q cp -r gs://sen1floods11/v1.1/splits/flood_handlabeled /content/splits
import os; print(len(os.listdir('/content/s1')), 'escenas S1')

## 2. Modelo: TerraTorch con backbone preentrenado
Configuración mínima de `terratorch` para segmentación binaria con entradas S1 (2 bandas). Cambiar `backbone` a `terramind_v1_base` o `clay_v1` según disponibilidad en la versión instalada; verificar con `terratorch.registry.BACKBONE_REGISTRY`.

In [ ]:
from terratorch.registry import BACKBONE_REGISTRY
print([n for n in BACKBONE_REGISTRY if 'terramind' in n.lower() or 'clay' in n.lower() or 'prithvi' in n.lower()][:20])

In [ ]:
config = '''
trainer:
  accelerator: gpu
  max_epochs: 20
  precision: 16-mixed
data:
  class_path: terratorch.datamodules.Sen1Floods11NonGeoDataModule
  init_args:
    data_root: /content
    batch_size: 8
    bands: [VV, VH]
    num_workers: 2
model:
  class_path: terratorch.tasks.SemanticSegmentationTask
  init_args:
    model_factory: EncoderDecoderFactory
    model_args:
      backbone: terramind_v1_base   # alternativa: clay_v1
      backbone_pretrained: true
      backbone_modalities: [S1GRD]
      decoder: UNetDecoder
      decoder_channels: [512, 256, 128, 64]
      num_classes: 2
    loss: ce
    ignore_index: -1
    freeze_backbone: false
    optimizer: AdamW
    lr: 1.0e-4
'''
open('/content/config.yaml','w').write(config)
!terratorch fit --config /content/config.yaml

## 3. Exportar a ONNX
La entrada esperada por `sarwatch.ml.segment.onnx_runner` es `(1, C, h, w)` float32 con canales normalizados; la salida `(1, 2, h, w)` logits. Guardar también `means`/`stds` usados para normalizar.

In [ ]:
import glob, torch, json
ckpt = sorted(glob.glob('/content/lightning_logs/**/*.ckpt', recursive=True))[-1]
from terratorch.tasks import SemanticSegmentationTask
task = SemanticSegmentationTask.load_from_checkpoint(ckpt).eval().cpu()
dummy = torch.zeros(1, 2, 512, 512)
torch.onnx.export(task.model, dummy, '/content/water_s1.onnx', input_names=['input'], output_names=['logits'], dynamic_axes={'input': {2: 'h', 3: 'w'}, 'logits': {2: 'h', 3: 'w'}}, opset_version=17)
json.dump({'bands': ['VV', 'VH'], 'note': 'normalización: completar con las medias/desvíos del datamodule'}, open('/content/water_s1.card.json', 'w'))
print('exportado')

## 4. Adaptación a NISAR (banda L)
1. Copiar `water_s1.onnx` a `models/` en el proyecto.
2. Mapear canales: HH→VV, HV→VH es una aproximación; documentarla.
3. Evaluar sin reentrenar con `scripts/evaluate_detection.py` (añadir el método ONNX) en los mismos bloques retenidos.
4. Si la brecha es grande, fine-tuning corto con las etiquetas Sentinel-2 de Pantanal (`data/labels/`) usando solo los bloques de entrenamiento de `sarwatch.ml.splits`.
5. Publicar F1/IoU por clase junto a LightGBM; entra al visor solo si gana.